# ☕ Dự báo giá cà phê Robusta tại Việt Nam

### Ứng dụng các mô hình học máy kết hợp hỗ trợ quyết định kinh doanh

## 1. Introduction


## 2. Import Libraries


## 3. Dataset Description

## 4. Nạp dữ liệu

> **Mục tiêu:** Đọc bộ dữ liệu VNCoffee và chuẩn bị biến giá cà phê cho các bước phân tích.

- Đọc file `vietnam_coffee_price_dataset.csv` bằng `pandas`.
- Tự tìm file trong thư mục dữ liệu cạnh notebook hoặc ở thư mục cha.
- Chuyển `date` sang kiểu ngày để thuận tiện cho phân tích chuỗi thời gian.
- Sử dụng `coffee_price_vnd_kg` làm **biến mục tiêu** — giá cà phê Robusta tính theo VND/kg.

**Lưu ý:** Nếu không tìm thấy file, notebook sẽ dừng và hiển thị hướng dẫn kiểm tra vị trí dữ liệu.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATASET_PATHS = [
    Path("VNCoffeePrice_mendeley_package_EN_v5 2/vietnam_coffee_price_dataset.csv"),
    Path("../VNCoffeePrice_mendeley_package_EN_v5 2/vietnam_coffee_price_dataset.csv"),
]
DATASET_PATH = next((path.resolve() for path in DATASET_PATHS if path.is_file()), None)
if DATASET_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy vietnam_coffee_price_dataset.csv. "
        "Hãy đặt thư mục VNCoffeePrice_mendeley_package_EN_v5 2 cạnh notebook."
    )

df = pd.read_csv(DATASET_PATH, encoding="utf-8-sig", parse_dates=["date"])
TARGET_COLUMN = "coffee_price_vnd_kg"
print(f"Dataset: {DATASET_PATH}")
print(f"Kích thước: {df.shape[0]:,} dòng x {df.shape[1]} cột")
display(df.head())

## 5. Khám phá dữ liệu (EDA)

> **Mục tiêu:** Nắm cấu trúc dữ liệu và quan sát đặc điểm giá cà phê trước khi tiền xử lý.

| Nội dung | Ý nghĩa |
|:--|:--|
| Tổng quan dữ liệu | Kiểm tra số dòng, kiểu dữ liệu và khoảng thời gian. |
| Phân chia tập | Xem số quan sát thuộc `train`, `val` và `test`. |
| So sánh theo tỉnh | Thống kê giá trung bình, trung vị và khoảng giá của từng tỉnh. |
| Diễn biến giá | Quan sát xu hướng giá theo thời gian trên biểu đồ đường. |
| Tương quan | Tìm các biến số có tương quan tuyến tính với giá cà phê. |

**Diễn giải:** Tương quan dùng để tham khảo mối liên hệ, không chứng minh quan hệ nhân quả.

In [ ]:
print("Kích thước và kiểu dữ liệu:")
df.info()

print("Khoảng thời gian:", df["date"].min().date(), "đến", df["date"].max().date())
print("Số quan sát theo tập dữ liệu:")
display(df["split"].value_counts().reindex(["train", "val", "test"]))

print("Số quan sát và giá cà phê (VND/kg) theo tỉnh:")
province_summary = df.groupby("province")[TARGET_COLUMN].agg(
    ["count", "mean", "median", "min", "max"]
).sort_values("mean", ascending=False)
display(province_summary.round(2))

price_by_date = df.pivot(index="date", columns="province", values=TARGET_COLUMN)
ax = price_by_date.plot(figsize=(14, 6), linewidth=1)
ax.set(title="Giá cà phê Robusta theo tỉnh", xlabel="Ngày", ylabel="Giá (VND/kg)")
ax.legend(title="Tỉnh")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
target_correlations = (
    df[numeric_columns].corr()[TARGET_COLUMN]
    .drop(TARGET_COLUMN)
    .sort_values(key=lambda values: values.abs(), ascending=False)
)
print("Các biến số có tương quan tuyệt đối cao nhất với giá cà phê:")
display(target_correlations.head(10).to_frame("correlation"))

## 6. Tiền xử lý dữ liệu

> **Mục tiêu:** Đưa dữ liệu về định dạng nhất quán, xử lý dữ liệu thiếu và bảo đảm mỗi quan sát là duy nhất.

Quy trình gồm ba bước:

1. **Làm sạch dữ liệu không nhất quán** — chuẩn hóa kiểu dữ liệu và loại các dòng không hợp lệ.
2. **Xử lý giá trị thiếu** — điền khuyết cho biến đầu vào; không tự tạo giá trị cho biến mục tiêu.
3. **Loại dữ liệu trùng lặp** — bảo đảm mỗi tỉnh có tối đa một bản ghi cho mỗi ngày.

**Bảo toàn thứ tự thời gian:** Giữ nguyên các nhãn `train` / `val` / `test`. Nếu cần điền khuyết, chỉ dùng thống kê từ tập `train` để hạn chế rò rỉ thông tin tương lai.

In [ ]:
print("Kích thước trước tiền xử lý:", df.shape)
print("Số giá trị thiếu theo cột trước tiền xử lý:")
display(df.isna().sum().sort_values(ascending=False).head(10))

### 6.1 Làm sạch dữ liệu không nhất quán

**Chuẩn hóa**
- Xóa khoảng trắng thừa trong tên cột và các trường phân loại.
- Chuẩn hóa nhãn tập dữ liệu; chuyển ngày và các cột số về đúng kiểu.

**Loại bỏ dòng không hợp lệ**
- Thiếu ngày, tỉnh, mặt hàng hoặc giá cà phê.
- Nhãn tập dữ liệu không thuộc `train`, `val` hoặc `test`.
- Giá mục tiêu không hữu hạn hoặc nhỏ hơn hay bằng 0.

Các cột bắt buộc được kiểm tra trước khi xử lý để phát hiện sớm dữ liệu sai cấu trúc.

In [ ]:
df = df.copy()
df.columns = df.columns.str.strip()

required_columns = {
    "date", "split", "province", "commodity", TARGET_COLUMN,
    "london_price_usd_ton", "usd_vnd_rate", "temperature_2m_mean",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Dataset thiếu các cột bắt buộc: {sorted(missing_columns)}")

df["date"] = pd.to_datetime(df["date"], errors="coerce")
non_numeric_columns = {"date", "split", "province", "commodity", "coord_source"}
numeric_columns = [column for column in df.columns if column not in non_numeric_columns]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

for column in ["split", "province", "commodity", "coord_source"]:
    if column in df.columns:
        df[column] = df[column].astype("string").str.strip()
df["split"] = df["split"].str.lower()

valid_splits = {"train", "val", "test"}
invalid_rows = (
    df["date"].isna()
    | df["province"].isna()
    | df["province"].eq("")
    | df["commodity"].isna()
    | df["commodity"].eq("")
    | ~df["split"].isin(valid_splits)
    | df[TARGET_COLUMN].isna()
    | ~np.isfinite(df[TARGET_COLUMN])
    | df[TARGET_COLUMN].le(0)
)
print(f"Số dòng không hợp lệ bị loại: {int(invalid_rows.sum())}")
df = df.loc[~invalid_rows].copy()
df = df.sort_values(["province", "date"]).reset_index(drop=True)
print("Số dòng sau khi chuẩn hóa ngày, nhãn và giá mục tiêu:", len(df))

### 6.2 Xử lý giá trị thiếu

Trước hết, kiểm tra số lượng giá trị thiếu ở từng cột. Cách xử lý phụ thuộc vào vai trò của cột:

| Loại dữ liệu | Cách xử lý |
|:--|:--|
| Giá cà phê mục tiêu | Loại dòng bị thiếu; không nội suy để tránh tạo nhãn giả. |
| Biến số đầu vào | Điền trung vị của tập `train` trong cùng tỉnh. |
| Không có trung vị theo tỉnh | Dùng trung vị của tập `train` trên toàn bộ dữ liệu. |

Sau khi điền khuyết, kiểm tra lại toàn bộ dữ liệu. Nếu vẫn còn giá trị thiếu, notebook sẽ báo lỗi để tránh âm thầm đưa dữ liệu chưa hoàn chỉnh vào mô hình.

In [ ]:
missing_before = df.isna().sum()
print("Giá trị thiếu trước khi xử lý:")
display(missing_before[missing_before.gt(0)])

# Không điền giá cà phê mục tiêu; loại bỏ dòng thiếu nhãn để tránh tạo giá dự báo giả.
missing_target = int(df[TARGET_COLUMN].isna().sum())
if missing_target:
    print(f"Loại {missing_target} dòng thiếu {TARGET_COLUMN}.")
    df = df.dropna(subset=[TARGET_COLUMN]).copy()

feature_columns = [
    column for column in df.select_dtypes(include=np.number).columns
    if column != TARGET_COLUMN
]
if feature_columns:
    train_rows = df["split"].eq("train")
    if not train_rows.any():
        raise ValueError("Không có dòng train để tính giá trị điền khuyết.")

    train_medians = df.loc[train_rows].groupby("province")[feature_columns].median()
    global_train_medians = df.loc[train_rows, feature_columns].median()
    for column in feature_columns:
        province_medians = df["province"].map(train_medians[column])
        df[column] = df[column].fillna(province_medians).fillna(global_train_medians[column])

remaining_missing = df.isna().sum()
remaining_missing = remaining_missing[remaining_missing.gt(0)]
if not remaining_missing.empty:
    raise ValueError(f"Vẫn còn giá trị thiếu chưa xử lý:\n{remaining_missing}")
print("Tổng số ô thiếu sau khi xử lý:", int(df.isna().sum().sum()))

### 6.3 Loại bỏ dữ liệu trùng lặp

Quá trình kiểm tra gồm hai cấp:

1. Loại các dòng trùng hoàn toàn.
2. Kiểm tra khóa **`(date, province)`** — mỗi tỉnh chỉ nên có một quan sát trong một ngày. Nếu khóa trùng, giữ bản ghi đầu tiên sau khi sắp xếp.

Cuối bước, sắp xếp dữ liệu theo tỉnh và ngày, rồi hiển thị kích thước, khoảng ngày và số dòng của từng tập để xác nhận kết quả.

In [ ]:
duplicate_rows = int(df.duplicated().sum())
print(f"Số bản ghi trùng hoàn toàn: {duplicate_rows}")
if duplicate_rows:
    df = df.drop_duplicates().copy()

# Mỗi tỉnh chỉ nên có một giá tại mỗi ngày; giữ bản ghi đầu sau khi đã sắp xếp.
duplicate_observations = int(df.duplicated(subset=["date", "province"]).sum())
print(f"Số bản ghi trùng khóa (date, province): {duplicate_observations}")
if duplicate_observations:
    df = df.drop_duplicates(subset=["date", "province"], keep="first").copy()

df = df.sort_values(["province", "date"]).reset_index(drop=True)
print("Kích thước cuối cùng:", df.shape)
print("Ngày:", df["date"].min().date(), "đến", df["date"].max().date())
print("Tập dữ liệu:", df["split"].value_counts().to_dict())
display(df.head())

## 7. Train-Test Split

## 8. Feature Scaling

## 9. Train Linear Regression Model 

## 10. Model Evaluation 

### 10.1 Performance Metrics

### 10.2 Actual vs Predicted Values

### 10.3 Actual vs Preducted Prices

## 11. Model Improvement